In [7]:
import kagglehub
import os
import shutil
from openai import OpenAI
import pandas as pd
import sqlite3

In [4]:
# Célula para inspeção estrutural (Saída em Texto Puro)
import pandas as pd
import os

# Pasta onde os dados estão localizados
data_dir = os.path.join(os.getcwd(), "data")

print("### Relatório Estrutural dos Arquivos (Formato Texto) ###\n")

for file_name in dict_rename.keys():
    full_path = os.path.join(data_dir, file_name)
    
    if os.path.exists(full_path):
        try:
            # low_memory=False evita avisos de tipos mistos durante a leitura inicial
            df_temp = pd.read_csv(full_path, low_memory=False)
            
            print(f"{'='*80}")
            print(f"ARQUIVO: {file_name}")
            print(f"{'='*80}")
            
            print("\n[INFO]")
            # O método info() imprime diretamente no console, não precisa de print()
            df_temp.info()
            
            print("\n[HEAD - Formato Texto]")
            # to_string() garante que a tabela saia como texto puro
            print(df_temp.head(3).to_string())
            
            print("\n[DESCRIBE - Formato Texto]")
            print(df_temp.describe().to_string())
            
            print("\n" + "."*80 + "\n")
            
        except Exception as e:
            print(f"Erro ao processar {file_name}: {e}")
    else:
        print(f"⚠️ Arquivo não encontrado no diretório: {file_name}")

### Relatório Estrutural dos Arquivos (Formato Texto) ###

ARQUIVO: Amazon Sale Report.csv

[INFO]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  object 
 2   Date                128975 non-null  object 
 3   Status              128975 non-null  object 
 4   Fulfilment          128975 non-null  object 
 5   Sales Channel       128975 non-null  object 
 6   ship-service-level  128975 non-null  object 
 7   Style               128975 non-null  object 
 8   SKU                 128975 non-null  object 
 9   Category            128975 non-null  object 
 10  Size                128975 non-null  object 
 11  ASIN                128975 non-null  object 
 12  Courier Status      122103 non-null  object 
 13  Qty                 128975 non-null

In [5]:
import pandas as pd
import numpy as np

def aplicar_tipagem_e_limpeza(df, file_name):
    """
    Aplica conversões baseadas na análise real dos arquivos:
    - Converte datas (formatos variados).
    - Limpa símbolos de moeda (₹, INR) e converte para float.
    - Garante integridade de colunas inteiras e booleanas.
    """
    
    # 1. Tratamento de Datas (Formatos detectados: '04-30-22', '06-05-21', etc.)
    colunas_data = ['date', 'received_date', 'DATE']
    for col in colunas_data:
        if col in df.columns:
            # dayfirst=False pois o padrão parece ser MM-DD-YY ou similar nos arquivos
            df[col] = pd.to_datetime(df[col], errors='coerce')

    # 2. Limpeza de Strings e Moedas (Trata '₹4.00', '616.56', 'INR')
    colunas_financeiras = [
        'amount', 'unit_rate', 'gross_amount', 'tp_price', 'tp_1_price', 'tp_2_price',
        'mrp_old', 'final_mrp_old', 'ajio_mrp', 'amazon_mrp', 'amazon_fba_mrp', 
        'flipkart_mrp', 'limeroad_mrp', 'myntra_mrp', 'paytm_mrp', 'snapdeal_mrp',
        'shiprocket_rate', 'increff_rate', 'received_amount', 'expense_amount'
    ]
    
    for col in colunas_financeiras:
        if col in df.columns:
            # Remove qualquer caractere que não seja número ou ponto
            df[col] = (df[col].astype(str)
                       .str.replace('₹', '', regex=False)
                       .str.replace('INR', '', regex=False)
                       .str.replace(',', '', regex=False)
                       .str.replace(r'[^\d.]', '', regex=True))
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3. Tratamento de Quantidades e Pesos (PCS e Weight estavam como object)
    colunas_quantitativas = ['qty', 'qty_pieces', 'stock_level', 'weight']
    for col in colunas_quantitativas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 4. Tratamento Específico para 'Expense IIGF.csv'
    # O arquivo possui uma linha de cabeçalho na index 0 ("Particular", "Amount")
    if file_name == "Expense IIGF.csv":
        df = df[df['received_amount'] != 'Amount'].copy()
        
    # 5. Booleano B2B
    if 'is_b2b' in df.columns:
        df['is_b2b'] = df['is_b2b'].astype(bool)

    return df

In [2]:
# 1. Download (Forçado para garantir que os arquivos venham)
path = kagglehub.dataset_download("thedevastator/unlock-profits-with-e-commerce-sales-data", force_download=True)

# 2. Destino
dest_dir = os.path.join(os.getcwd(), "data")

# 3. Transferência Direta
# dirs_exist_ok=True: Copia mesmo se a pasta 'data' já existir
# copy_function=shutil.copy2: Garante a cópia dos arquivos com metadados
try:
    shutil.copytree(path, dest_dir, dirs_exist_ok=True)
    print(f"✅ Sucesso! Arquivos em: {dest_dir}")
    print(f"Conteúdo: {os.listdir(dest_dir)}")
except Exception as e:
    print(f"❌ Erro: {e}")

100%|██████████| 6.33M/6.33M [00:01<00:00, 3.76MB/s]

Extracting files...


✅ Sucesso! Arquivos em: c:\Users\flavio.bezerra\OneDrive\Documentos\07. Processos Seletivos\Boticário\data
Conteúdo: ['Amazon Sale Report.csv', 'Cloud Warehouse Compersion Chart.csv', 'Expense IIGF.csv', 'International sale Report.csv', 'May-2022.csv', 'P  L March 2021.csv', 'Sale Report.csv']


In [2]:
dict_rename = {
  "Amazon Sale Report.csv": {
    "index": "id",
    "Order ID": "order_id",
    "Date": "date",
    "Status": "status",
    "Fulfilment": "fulfilment",
    "Sales Channel ": "sales_channel",
    "ship-service-level": "ship_service_level",
    "Style": "style",
    "SKU": "sku",
    "Category": "category",
    "Size": "size",
    "ASIN": "asin",
    "Courier Status": "courier_status",
    "Qty": "qty",
    "currency": "currency",
    "Amount": "amount",
    "ship-city": "ship_city",
    "ship-state": "ship_state",
    "ship-postal-code": "ship_postal_code",
    "ship-country": "ship_country",
    "promotion-ids": "promotion_ids",
    "B2B": "is_b2b",
    "fulfilled-by": "fulfilled_by",
    "Unnamed: 22": "is_fulfilled"
  },
  "Sale Report.csv": {
    "index": "id",
    "SKU Code": "sku",
    "Design No.": "design_no",
    "Stock": "stock_level",
    "Category": "category",
    "Size": "size",
    "Color": "color"
  },
  "P  L March 2021.csv": {
    "index": "id",
    "Sku": "sku",
    "Style Id": "style_id",
    "Catalog": "catalog",
    "Category": "category",
    "Weight": "weight",
    "TP 1": "tp_1_price",
    "TP 2": "tp_2_price",
    "MRP Old": "mrp_old",
    "Final MRP Old": "final_mrp_old",
    "Ajio MRP": "ajio_mrp",
    "Amazon MRP": "amazon_mrp",
    "Amazon FBA MRP": "amazon_fba_mrp",
    "Flipkart MRP": "flipkart_mrp",
    "Limeroad MRP": "limeroad_mrp",
    "Myntra MRP": "myntra_mrp",
    "Paytm MRP": "paytm_mrp",
    "Snapdeal MRP": "snapdeal_mrp"
  },
  "May-2022.csv": {
    "index": "id",
    "Sku": "sku",
    "Style Id": "style_id",
    "Catalog": "catalog",
    "Category": "category",
    "Weight": "weight",
    "TP": "tp_price",
    "MRP Old": "mrp_old",
    "Final MRP Old": "final_mrp_old",
    "Ajio MRP": "ajio_mrp",
    "Amazon MRP": "amazon_mrp",
    "Amazon FBA MRP": "amazon_fba_mrp",
    "Flipkart MRP": "flipkart_mrp",
    "Limeroad MRP": "limeroad_mrp",
    "Myntra MRP": "myntra_mrp",
    "Paytm MRP": "paytm_mrp",
    "Snapdeal MRP": "snapdeal_mrp"
  },
  "International sale Report.csv": {
    "index": "id",
    "DATE": "date",
    "Months": "month",
    "CUSTOMER": "customer_name",
    "Style": "style",
    "SKU": "sku",
    "Size": "size",
    "PCS": "qty_pieces",
    "RATE": "unit_rate",
    "GROSS AMT": "gross_amount"
  },
  "Expense IIGF.csv": {
    "index": "id",
    "Recived Amount": "received_date",
    "Unnamed: 1": "received_amount",
    "Expance": "expense_description",
    "Unnamed: 3": "expense_amount"
  },
  "Cloud Warehouse Compersion Chart.csv": {
    "index": "id",
    "Shiprocket": "service_type",
    "Unnamed: 1": "shiprocket_rate",
    "INCREFF": "increff_rate"
  }
}

dict_business_names = {
    "Amazon Sale Report.csv": "vendas_marketplace_amazon",
    "International sale Report.csv": "vendas_e_exportacoes_internacionais",
    "P  L March 2021.csv": "performance_financeira_e_custos_marco_2021",
    "May-2022.csv": "performance_financeira_e_custos_maio_2022",
    "Sale Report.csv": "gestao_de_inventario_e_sku",
    "Expense IIGF.csv": "controle_de_despesas_e_fluxo_de_caixa",
    "Cloud Warehouse Compersion Chart.csv": "comparativo_de_custos_de_operadores_logisticos"
}

In [8]:
# 1. Configurar caminhos e conexão
import os
data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(data_dir, exist_ok=True)
db_path = os.path.join(data_dir, 'ecommerce_analytics.db')
conn = sqlite3.connect(db_path)

for file_name in dict_rename.keys():
    full_path = os.path.join(data_dir, file_name)
    
    if os.path.exists(full_path):
        # Para arquivos com lixo no topo ou formatos específicos
        df = pd.read_csv(full_path, low_memory=False)
        
        # Primeiro renomeamos para usar os nomes padronizados na função de limpeza
        df = df.rename(columns=dict_rename[file_name])
        
        # Aplicamos a tipagem robusta
        df = aplicar_tipagem_e_limpeza(df, file_name)
        
        # Nome da tabela no banco
        table_name = dict_business_names.get(file_name, file_name.replace('.csv', '').lower())
        
        # Salva no SQLite
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"✅ Tabela '{table_name}' salva e tipada!")

conn.close()

C:\Users\flavio.bezerra\AppData\Local\Temp\ipykernel_7568\4174814223.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')


✅ Tabela 'vendas_marketplace_amazon' salva e tipada!
✅ Tabela 'gestao_de_inventario_e_sku' salva e tipada!
✅ Tabela 'performance_financeira_e_custos_marco_2021' salva e tipada!
✅ Tabela 'performance_financeira_e_custos_maio_2022' salva e tipada!


C:\Users\flavio.bezerra\AppData\Local\Temp\ipykernel_7568\4174814223.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')


✅ Tabela 'vendas_e_exportacoes_internacionais' salva e tipada!
✅ Tabela 'controle_de_despesas_e_fluxo_de_caixa' salva e tipada!
✅ Tabela 'comparativo_de_custos_de_operadores_logisticos' salva e tipada!


C:\Users\flavio.bezerra\AppData\Local\Temp\ipykernel_7568\4174814223.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')


In [ ]:
import os
import sqlite3
import pandas as pd

import os
data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(data_dir, exist_ok=True)
db_path = os.path.join(data_dir, 'ecommerce_analytics.db')
conn = sqlite3.connect(db_path)

query = """
SELECT *
FROM comparativo_de_custos_de_operadores_logisticos
limit 5
"""

df = pd.read_sql(query, conn)
print(df)

   id                      service_type  shiprocket_rate  increff_rate
0   0                             Heads              NaN           NaN
1   1     Inbound (Fresh Stock and RTO)              4.0          4.00
2   2                          Outbound              7.0         11.00
3   3                   Storage Fee/Cft             25.0          0.15
4   4  Customer Return with Detailed QC              6.0         15.50


In [9]:
import os
import sqlite3
import pandas as pd

# Configuração do caminho do banco (baseado no seu notebook de ingestão)
import os
data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(data_dir, exist_ok=True)
db_path = os.path.join(data_dir, 'ecommerce_analytics.db')
conn = sqlite3.connect(db_path)

# 1. Busca os nomes de todas as tabelas existentes no banco
query_tabelas = "SELECT name FROM sqlite_master WHERE type='table';"
tabelas = pd.read_sql(query_tabelas, conn)['name'].tolist()

print("### Verificação de Tipagem Pós-Ingestão ###\n")

for tabela in tabelas:
    print(f"{'='*80}")
    print(f"TABELA: {tabela}")
    print(f"{'='*80}")
    
    # Lê os dados da tabela
    query = f"SELECT * FROM {tabela}"
    df_verificacao = pd.read_sql(query, conn)
    
    # Exibe o info para comprovar os Dtypes (float64, int64, datetime64, etc.)
    print("\n[ESTRUTURA DE TIPOS - INFO]")
    df_verificacao.info()
    
    print("\n[AMOSTRA DE DADOS - HEAD]")
    # Usando to_string() para facilitar a cópia para o chat
    print(df_verificacao.head(3).to_string())
    
    print("\n" + "."*80 + "\n")

conn.close()

### Verificação de Tipagem Pós-Ingestão ###

TABELA: vendas_marketplace_amazon

[ESTRUTURA DE TIPOS - INFO]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  128975 non-null  int64  
 1   order_id            128975 non-null  object 
 2   date                128975 non-null  object 
 3   status              128975 non-null  object 
 4   fulfilment          128975 non-null  object 
 5   sales_channel       128975 non-null  object 
 6   ship_service_level  128975 non-null  object 
 7   style               128975 non-null  object 
 8   sku                 128975 non-null  object 
 9   category            128975 non-null  object 
 10  size                128975 non-null  object 
 11  asin                128975 non-null  object 
 12  courier_status      122103 non-null  object 
 13  qty                 128975